# GPT-2 From Scratch in PyTorch

We'll build a full GPT-2 (124M) architecture from scratch, then train a smaller version on a small text corpus so it runs on any laptop / Colab.

**Sections:**
1. Imports & config
2. Model components (attention, MLP, block, GPT)
3. Data & tokenizer
4. Training loop
5. Sampling / generation
6. (Optional) Load OpenAI GPT-2 weights

In [3]:
import math
import os
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)

device: cpu
torch: 2.15.0+cpu


We only need PyTorch and the standard library. device is auto-detected; everything later moves with .to(device).

**Code Cell 2 — Config**

In [4]:
@dataclass
class GPTConfig:
    block_size: int = 1024      # context length
    vocab_size: int = 50257     # GPT-2 BPE vocab (we override for the demo)
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True

Explanation:

These are exactly OpenAI's gpt2 (124M) hyperparameters. Later we override them with a smaller config so training is fast on a single device.

**Code Cell 3 — Causal Self-Attention**

In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # causal mask: token t can only attend to positions <= t
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size))
                 .view(1, 1, config.block_size, config.block_size),
            persistent=False,
        )

    def forward(self, x):
        B, T, C = x.size()

        # qkv: (B, T, 3C) -> (B, T, 3, nh, hd) -> split -> (B, nh, T, hd)
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # scaled dot-product attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v                                  # (B, nh, T, hd)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

Explanation:

Standard multi-head self-attention with one crucial change: the mask. The bias buffer holds a lower-triangular matrix of ones. Where it's zero, we set scores to -inf before softmax, so token t cannot attend to the future — this is what makes the model causal (autoregressive).



**Code Cell 4 — MLP**

In [6]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return self.dropout(x)

Explanation:

The transformer's FFN. Expands 4×, applies GELU (GPT-2 used tanh approximation), projects back down.

**Code Cell 5 — Transformer Block**

In [7]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))     # residual + pre-LN
        x = x + self.mlp(self.ln_2(x))
        return x

Explanation:

Pre-LayerNorm architecture: normalization happens before the sublayer, not after. This is what made GPT-2 trainable at depth 12+ without warm-up tricks. Every sublayer output is added to the residual stream.

**Code Cell 6 — Full GPT model**

In [8]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),   # token emb
            wpe = nn.Embedding(config.block_size, config.n_embd),   # position emb
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight tying
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size

        pos = torch.arange(T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])   # only last position needed
            return logits, None

Explanation:

Three details worth noting:

Weight tying (wte.weight = lm_head.weight) — the input embedding and output projection share parameters. This is why the 124M model isn't ~163M.

Scaled init for c_proj weights (0.02 / sqrt(2 * n_layer)) — keeps signal variance stable through deep residual streams.

Inference shortcut — during sampling we only compute logits for the last token, since we only need the next-token distribution.

**Code Cell 7 — Sanity check**

In [10]:
cfg = GPTConfig(block_size=64, vocab_size=100, n_layer=2, n_head=4, n_embd=64)
m = GPT(cfg)
x = torch.randint(0, 100, (2, 16))
logits, loss = m(x, x)
print("logits:", logits.shape, "loss:", loss.item())
print("params:", m.num_params())

logits: torch.Size([2, 16, 100]) loss: 3.9306693077087402
params: 110592


Explanation:

Small model, tiny batch. loss should be around ln(100) ≈ 4.6 since it's random init. If this runs, the whole graph is correct — all other bugs are in data or training.

**Code Cell 8 — Data: char-level tokenizer (demo)**

In [11]:
text = (
    "Once upon a time there was a curious programmer who wanted to build "
    "a language model from scratch. They wrote attention, trained on small "
    "text, and watched the loss go down. The end was never really the end. "
) * 500

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

BLOCK = 128
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]

print("vocab:", len(chars), "| total tokens:", len(data))

vocab: 27 | total tokens: 104000


Explanation:

Character-level tokenization keeps this demo dependency-free. For real GPT-2 later, swap this out for tiktoken — we'll show that at the end.


**Code Cell 9 — Batching**

In [12]:
def get_batch(split, batch_size=32):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK - 1, (batch_size,))
    x = torch.stack([d[i : i + BLOCK] for i in ix])
    y = torch.stack([d[i + 1 : i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", 4)
print("x:", xb.shape, "y:", yb.shape)

x: torch.Size([4, 128]) y: torch.Size([4, 128])


Explanation:

The target y is the input shifted by one — classic next-token prediction. Each sample is a random contiguous slice of the corpus.



**Code Cell 10 — Build the model**

In [13]:
config = GPTConfig(
    block_size=BLOCK,
    vocab_size=len(chars),
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1,
)
model = GPT(config).to(device)
print(f"model params: {model.num_params()/1e6:.2f}M")

model params: 10.71M


Explanation:

This is a ~10M-parameter GPT-2-style model. Enough to see real learning on a small corpus. Set n_layer=12, n_head=12, n_embd=768 to match the real 124M model (only feasible with a GPU and a real dataset)

**Code Cell 11 — Optimizer**

In [14]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1
)

Explanation:

GPT-2's recipe: AdamW with betas=(0.9, 0.95), weight_decay=0.1, LR 3e-4. In a full run you'd add a warmup + cosine decay schedule.

**Code Cell 12 — Train (or overfit first)**

In [15]:
    # --- overfit one batch to validate the pipeline ---
xb, yb = get_batch("train", 8)
model.train()
for step in range(50):
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"overfit step {step}: loss {loss.item():.4f}")

overfit step 0: loss 3.4057
overfit step 10: loss 2.7696
overfit step 20: loss 2.1377
overfit step 30: loss 1.6509
overfit step 40: loss 1.2284


Explanation:

Always overfit a single batch first. If loss doesn't drop to near 0 here, there's a bug in the model or data pipeline — do not continue. This is the single most valuable debugging step.

**Code Cell 13 — Full training loop**

In [16]:
max_steps = 1500
eval_every = 250
batch_size = 32

model.train()
t0 = time.time()
for step in range(max_steps):
    xb, yb = get_batch("train", batch_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch("val", batch_size)
            _, val_loss = model(xv, yv)
        model.train()
        dt = time.time() - t0
        print(f"step {step:4d} | train {loss.item():.4f} | "
              f"val {val_loss.item():.4f} | {dt:.1f}s")

step    0 | train 2.2462 | val 2.2924 | 2.2s
step  250 | train 0.0453 | val 0.0286 | 410.4s
step  500 | train 0.0269 | val 0.0235 | 784.4s
step  750 | train 0.0270 | val 0.0242 | 1168.2s
step 1000 | train 0.0285 | val 0.0227 | 1554.7s
step 1250 | train 0.0297 | val 0.0220 | 1931.5s
step 1499 | train 0.0220 | val 0.0256 | 2311.7s


Explanation:

Gradient clipping at norm 1.0 is standard for GPT training. The gap between train and val loss tells you whether you're overfitting. On this small corpus, val should track train closely for the first thousand steps.

**Code Cell 14 — Sampling**

In [17]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

Explanation:

Temperature scales logits before softmax (higher = more random, lower = more deterministic). Top-k keeps only the k most likely tokens, which prevents rare nonsense tokens from sneaking in. This is the same algorithm GPT-2 uses at inference.



**Code Cell 15 — Generate sample text**

In [18]:
model.eval()
start = torch.tensor([[stoi[text[0]]]], dtype=torch.long, device=device)
out = generate(model, start, max_new_tokens=400, temperature=0.8, top_k=40)
print("".join(itos[i] for i in out[0].tolist()))

Once upon a time there was a curious programmer who wanted to build a language model from scratch. They wrote attention, trained on small text, and watched the loss go down. The end was never really the end. Once upon a time there was a curious programmer who wanted to build a language model from scratch. They wrote attention, trained on small text, and watched the loss go down. The end was never r


**Code Cell 16 — Save model**

In [19]:
torch.save(
    {"model": model.state_dict(), "config": config},
    "gpt2_scratch.pt",
)
print("saved gpt2_scratch.pt")

saved gpt2_scratch.pt
